In [3]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

In [26]:
def ma_rsi(timeframe, start, end, st_ma, lt_ma, tp, sl):
    df = vbt.CCXTData.download('BTC/USDT', timeframe=timeframe, start = start, end = end).get()

    df['st_ma'] = ta.trend.SMAIndicator(df['Close'], st_ma).sma_indicator()
    df['lt_ma'] = ta.trend.SMAIndicator(df['Close'], lt_ma).sma_indicator()

    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()

    cond_ma = [ df['st_ma'] > df['lt_ma'], df['st_ma'] < df['lt_ma'] ]
    df['signal_ma'] = np.select(cond_ma, [1,-1], 0)

    df['signal_rsi'] = np.where(df['rsi'].between(0,50), 1, -1)

    cond_go = [ (df['signal_rsi'] == 1) & (df['signal_ma'] == 1), 
               (df['signal_rsi'] == -1) & (df['signal_ma'] == -1)  ]
    df['go'] =np.select(cond_go, [1, -1], 0)

    pf = vbt.Portfolio.from_signals(df['Close'], 
                                    entries = (df['go'] == 1), 
                                    exits = (df['go'] == -1), 
                                    fees = 0.0006,
                                    tp_stop = tp,
                                    sl_stop = sl,
                                    short_entries = (df['go'] == -1),
                                    short_exits = (df['go'] == 1)                                                         
                                   )

    return pf.stats()['Total Return [%]'], pf.stats()['Win Rate [%]'], pf.stats()['Max Drawdown [%]']

In [17]:
ma_rsi('4h', '1 month ago', 20, 40, 0.09, 0.02)
ma_rsi('4h', '1 month ago', 10, 40, 0.09, 0.02)
ma_rsi('4h', '1 month ago', 20, 30, 0.09, 0.02)
ma_rsi('4h', '1 month ago', 20, 40, 0.08, 0.02)

0it [00:00, ?it/s]

(-6.23026297818393, 60.0, 9.254252849650285)

In [19]:
list_timeframe = ['4h', '1d']
list_st_ma = [10, 20]
list_lt_ma = np.arange(start=30, stop=50, step=10)
list_tp = np.arange(start = 0.05, stop = 0.1, step = 0.01)
list_sl = np.arange(start = 0.01, stop = 0.04, step = 0.01)

In [21]:
empty = []
for timeframe in list_timeframe:
    for st_ma in list_st_ma:
        for lt_ma in list_lt_ma:
            for tp in list_tp:
                for sl in list_sl:
                    (total_return, win_rate, mdd) = ma_rsi(timeframe, '6 months ago', st_ma, lt_ma, tp, sl)
                    dct = {}
                    dct['timeframe'] = timeframe
                    dct['st_ma'] = st_ma
                    dct['lt_ma'] = lt_ma
                    dct['tp'] = tp
                    dct['sl'] = sl
                    dct['total_return'] = total_return
                    dct['win_rate'] = win_rate
                    dct['max_drawdown'] = mdd
                    empty.append(dct)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

... (110 outputs left out)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [24]:
df_comparison = pd.DataFrame(empty)
df_comparison

,timeframe,st_ma,lt_ma,tp,sl,total_return,win_rate,max_drawdown
0,4h,10,30,0.05,0.01,-17.407327,33.802817,22.366317
1,4h,10,30,0.05,0.02,-0.024758,51.785714,20.896236
2,4h,10,30,0.05,0.03,3.763404,58.490566,16.134936
3,4h,10,30,0.06,0.01,-26.726200,30.303030,27.599439
4,4h,10,30,0.06,0.02,-11.659738,48.076923,23.782054
...,...,...,...,...,...,...,...,...
115,1d,20,40,0.08,0.02,20.417104,50.000000,16.217528
116,1d,20,40,0.08,0.03,6.674415,42.857143,18.501501
117,1d,20,40,0.09,0.01,9.396671,37.500000,17.804371
118,1d,20,40,0.09,0.02,22.435729,50.000000,16.217528


In [25]:
df_comparison.sort_values(['total_return'], ascending=False).head(10)

,timeframe,st_ma,lt_ma,tp,sl,total_return,win_rate,max_drawdown
118,1d,20,40,0.09,0.02,22.435729,50.000000,16.217528
109,1d,20,40,0.06,0.02,21.635741,55.555556,16.217528
106,1d,20,40,0.05,0.02,21.631448,55.555556,16.217528
115,1d,20,40,0.08,0.02,20.417104,50.000000,16.217528
112,1d,20,40,0.07,0.02,15.929397,50.000000,16.217528
88,1d,10,40,0.09,0.02,13.252911,42.857143,16.217528
79,1d,10,40,0.06,0.02,12.217846,50.000000,16.217528
76,1d,10,40,0.05,0.02,12.214942,50.000000,16.217528
85,1d,10,40,0.08,0.02,11.098584,42.857143,16.217528
117,1d,20,40,0.09,0.01,9.396671,37.500000,17.804371


In [28]:
a = np.asarray(ma_rsi('1d', '12 months ago', '6 months ago',  20, 40, 0.09, 0.02))
b = np.asarray(ma_rsi('1d', '6 months ago', '3 months ago',  20, 40, 0.09, 0.02))
c = np.asarray(ma_rsi('1d', '3 months ago', '1 day ago',  20, 40, 0.09, 0.02))
(a+b+c)/3

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

array([ 7.69119244, 54.7008547 ,  8.53110359])

In [39]:
a = np.asarray(ma_rsi('1d', '12 months ago', '9 months ago',  20, 40, 0.09, 0.02))
b = np.asarray(ma_rsi('1d', '8 months ago', '7 months ago',  20, 40, 0.09, 0.02))
c = np.asarray(ma_rsi('1d', '3 months ago', '1 day ago',  20, 40, 0.09, 0.02))
(a+b+c)/3

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

array([2.32443131,        nan,        nan])

In [40]:
a

array([-1.6530248 , 33.33333333, 12.25167489])

In [41]:
b

array([ 0., nan, nan])

In [42]:
a = np.asarray(ma_rsi('1d', '36 months ago', '1 day ago',  20, 40, 0.09, 0.02))
b = np.asarray(ma_rsi('1d', '24 months ago', '1 day ago',  20, 40, 0.09, 0.02))
c = np.asarray(ma_rsi('1d', '12 months ago', '1 day ago',  20, 40, 0.09, 0.02))
(a+b+c)/3

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

array([-17.85414184,  29.90228371,  42.33551039])